# Process the Customers Data 
1. Ingest the data into the data lakehouse - bronze_customers
2. Perform data quality checks and transform the data as required - silver_customers_clean
3. Apply changes to the Customers data - silver_customers


## 1.Ingest the data into the data lakehouse - bronze_customers 

In [0]:
%sql
USE CATALOG circuitbox;
CREATE VOLUME IF NOT EXISTS circuitbox.lakehouse.schemas;
USE SCHEMA lakehouse;

CREATE TABLE IF NOT EXISTS bronze_customers (
  customer_id BIGINT,
  created_date STRING,
  customer_name STRING,
  date_of_birth STRING,
  email STRING,
  telephone STRING,
  _rescued_data STRING, 
  input_file_path STRING,
  ingestion_timestamp TIMESTAMP
)
USING DELTA


## 2. Perform data quality checks and transfrom the data as required - silver_customers_clean

![image_1778863233022.png](./imagens/image_1778863233022.png "image_1778863233022.png")

In [0]:
%sql
SELECT * 
FROM (circuitbox.lakehouse.silver_customers_clean);

![image_1778863293371.png](./imagens/image_1778863293371.png "image_1778863293371.png")

## 3. Apply changes to the Customers data - silver_customers

![image_1779109731730.png](./imagens/image_1779109731730.png "image_1779109731730.png")

In [0]:
%sql 
CREATE OR REFRESH STREAMING TABLE silver_customers 
COMMENT  'SCD Type 1 customer data'
TBLPROPERTIES ('quality'= 'silver')

In [0]:
%sql
APPLY CHANGES INTO LIVE.silver_customers
FROM STREAM(LIVE.silver_customers_clean)
KEYS (customer_id)
SEQUENCE BY created_date
STORED AS SCD TYPE 1;
